# Notebook 7: `facebook/omniASR-CTC-300M`

Family: `omni_asr` · **Expected to FAIL** on Kaggle (DECISIONS.md §D — fairseq2n CUDA-13 wheel vs Kaggle's CUDA 12.x).

Still run the notebook end to end: the transcription cell will mark the model FAILED and produce an empty `predictions/facebook__omniASR-CTC-300M.csv` with a header row. Upload it to notebook 10 like the others — the judge will surface it as FAILED in the master JSON.


In [ ]:
# === Clone the benchmark repo ===========================================
# Kaggle paths: /kaggle/working is the writable home. We clone the project
# repo there and cd into it so the relative paths used by src/ resolve.

!apt-get -qq install -y libsndfile1

GITHUB_REPO_URL = "https://github.com/Mainul/banglish-stt-benchmark.git"
REPO_DIR = "/kaggle/working/banglish-stt-benchmark"

import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# === Install omnilingual-asr (expected to FAIL) =========================
# DECISIONS.md §D: omnilingual-asr depends on fairseq2n which ships only
# a CUDA-13 wheel. Kaggle provides CUDA 12.x → libcudart.so.13 missing.
# The plan locks the model list, so we install + try anyway; the
# transcription cell catches the load error and writes a FAILED status.
#
# If you happen to be on a CUDA-13 runtime, this might actually work —
# the rest of the notebook is identical to the other model notebooks.

!pip install -q omnilingual-asr || true
!pip install -q transformers==4.44.2 accelerate==0.34.2 librosa==0.10.2.post1 \
    soundfile==0.12.1 pandas==2.2.3 pyyaml==6.0.2 tqdm==4.67.1 requests==2.32.3


In [ ]:
# === Build the 50-clip manifest =========================================
# Downloads OpenSLR-104 if not already cached, extracts, parses transcripts,
# computes length + code-switch density, and stratified-samples 50 clips.
#
# CRITICAL: every notebook in this benchmark calls this with seed=42 so
# they all see the EXACT SAME 50 clips. Do not change the seed.

from src.utils import setup_logging, set_seeds
from src.data_prep import build_manifest

setup_logging("logs")
set_seeds(42)

manifest_path = build_manifest(
    raw_dir="data/raw",
    manifest_path="data/manifest.csv",
    n_samples=50,
    seed=42,
)
print("manifest:", manifest_path)


In [ ]:
# === Transcribe with this model =========================================
# Reads the spec for THIS notebook's model from config/models.yaml,
# then runs the 50-clip loop. Resumable: rerun this cell to pick up
# where it stopped (clips already in the CSV are skipped).

from src.utils import read_yaml
from src.transcribe import run_single_model

MODEL_ID = "facebook/omniASR-CTC-300M"
spec = next(s for s in read_yaml("config/models.yaml")["models"] if s["id"] == MODEL_ID)

failure = run_single_model(
    spec,
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    device="cuda",
)
print("FAILED:", failure) if failure else print("OK — see predictions/")


In [ ]:
# === Ensure the predictions CSV exists (even if FAILED) =================
# If the load failed, run_single_model returned a failure dict but did not
# write a CSV. Write a header-only file so the judge notebook sees the
# expected slug and marks the model FAILED via its existing missing-model
# path.

from pathlib import Path
from src.transcribe import PREDICTION_COLUMNS

csv_path = Path("predictions/facebook__omniASR-CTC-300M.csv")
if not csv_path.exists():
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_text(",".join(PREDICTION_COLUMNS) + "\n", encoding="utf-8")
    print(f"Wrote header-only {csv_path} (model FAILED).")
else:
    print(f"{csv_path} already exists ({csv_path.stat().st_size} bytes).")


In [ ]:
# === Download the predictions CSV =======================================
# Right-click the link below and save to your local machine. You will
# upload all 9 CSVs to the judge notebook (notebook 10) at the end.
#
# File:   predictions/facebook__omniASR-CTC-300M.csv
# Schema: clip_id, model_id, predicted_text, latency_sec, decoder_variant

from pathlib import Path
import pandas as pd
from IPython.display import FileLink, display

csv_path = Path("predictions/facebook__omniASR-CTC-300M.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"{len(df)} rows in {csv_path}")
    display(df.head())
    display(FileLink(str(csv_path)))
else:
    print(f"NOT FOUND: {csv_path} — transcription cell failed or was skipped.")
